## Apex Wealth Data Pipeline

In [4]:
!pip install request
!pip install python-dotenv

ERROR: Could not find a version that satisfies the requirement request (from versions: none)
ERROR: No matching distribution found for request

[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: C:\Users\NED\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.0 -> 26.0
[notice] To update, run: C:\Users\NED\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [5]:
import requests
import pandas as pd 
from dotenv import load_dotenv
import os
import psycopg2
from sqlalchemy import create_engine

## Extraction

In [6]:
load_dotenv
api_key = os.getenv('API_KEY')

api_key

'0bb27de9424a464ba47cd89911e640de'

In [7]:
# define url endpoint
symbol = 'AAPL'
url = f'https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=4'

In [8]:
# make a request to the API 
response = requests.get(url)
response.status_code

200

In [9]:
# get data in json format
data = response.json()
data

{'meta': {'symbol': 'AAPL',
  'interval': '1min',
  'currency': 'USD',
  'exchange_timezone': 'America/New_York',
  'exchange': 'NASDAQ',
  'mic_code': 'XNGS',
  'type': 'Common Stock'},
 'values': [{'datetime': '2026-01-30 15:59:00',
   'open': '259.88000',
   'high': '260.06',
   'low': '258',
   'close': '259.48001',
   'volume': '1973068'},
  {'datetime': '2026-01-30 15:58:00',
   'open': '259.60001',
   'high': '259.93',
   'low': '259.51501',
   'close': '259.87000',
   'volume': '683270'},
  {'datetime': '2026-01-30 15:57:00',
   'open': '259.87000',
   'high': '259.88',
   'low': '259.36499',
   'close': '259.57001',
   'volume': '574705'},
  {'datetime': '2026-01-30 15:56:00',
   'open': '259.82999',
   'high': '260.040009',
   'low': '259.66000',
   'close': '259.85999',
   'volume': '375311'}],
 'status': 'ok'}

In [10]:
# tranfer data to a dataframe
def transform_data(data):
    
    
    # extract values from json
    time_series = data['values']
    
    #convert to dataframe
    df = pd.DataFrame(time_series)
    
    # convert columns to appropriate data type
    df['datetime'] = pd.to_datetime(df['datetime'])
    
    df = df.astype({
        'open':'float',
        'high':'float',
        'low':'float',
        'close':'float',
        'volume':'int'
    })
    return df

In [11]:
df = transform_data(data)
df

,datetime,open,high,low,close,volume
0,2026-01-30 15:59:00,259.88000,260.060000,258.00000,259.48001,1973068
1,2026-01-30 15:58:00,259.60001,259.930000,259.51501,259.87000,683270
2,2026-01-30 15:57:00,259.87000,259.880000,259.36499,259.57001,574705
3,2026-01-30 15:56:00,259.82999,260.040009,259.66000,259.85999,375311


### Extract from multiple symbols

In [12]:
symbols =['AAPL','MSFT','GOOGL','AMZN']
all_data = []

def fetch_data(symbol,api_key):
    url = f'https://api.twelvedata.com/time_series?symbol={symbol}&interval=1min&apikey={api_key}&outputsize=4'
    response = requests.get(url)
    response.raise_for_status() # raise an error for bad responses
    data = response.json()
    
    #check if the api response is ok
    if data['status'] != 'ok':
        print(f"error with {symbol}: {data['message']}")
    return data
    

In [13]:
symbols =['AAPL','MSFT','GOOGL','AMZN']

for symbol in symbols:
    data = fetch_data(symbol,api_key)
    df = transform_data(data)
    df['sym'] = symbol # add a column for the symbol
    all_data.append(df)

In [14]:
all_data

[             datetime       open        high        low      close   volume  \
 0 2026-01-30 15:59:00  259.88000  260.060000  258.00000  259.48001  1973068   
 1 2026-01-30 15:58:00  259.60001  259.930000  259.51501  259.87000   683270   
 2 2026-01-30 15:57:00  259.87000  259.880000  259.36499  259.57001   574705   
 3 2026-01-30 15:56:00  259.82999  260.040009  259.66000  259.85999   375311   
 
     sym  
 0  AAPL  
 1  AAPL  
 2  AAPL  
 3  AAPL  ,
              datetime       open       high        low      close  volume  \
 0 2026-01-30 15:59:00  429.90009  430.42001  429.49011  430.39001  846171   
 1 2026-01-30 15:58:00  429.68500  429.94000  429.35000  429.90500  346721   
 2 2026-01-30 15:57:00  429.67001  429.98001  429.53000  429.69000  293137   
 3 2026-01-30 15:56:00  429.34000  429.70639  429.23999  429.67999  283368   
 
     sym  
 0  MSFT  
 1  MSFT  
 2  MSFT  
 3  MSFT  ,
              datetime        open       high        low       close  volume  \
 0 2026-01-30 

In [15]:
# using concat - joining or stacking all the data together
all_df = pd.concat(all_data,ignore_index=True)

In [16]:
all_df

,datetime,open,high,low,close,volume,sym
0,2026-01-30 15:59:00,259.880000,260.060000,258.00000,259.480010,1973068,AAPL
1,2026-01-30 15:58:00,259.600010,259.930000,259.51501,259.870000,683270,AAPL
2,2026-01-30 15:57:00,259.870000,259.880000,259.36499,259.570010,574705,AAPL
3,2026-01-30 15:56:00,259.829990,260.040009,259.66000,259.859990,375311,AAPL
4,2026-01-30 15:59:00,429.900090,430.420010,429.49011,430.390010,846171,MSFT
5,2026-01-30 15:58:00,429.685000,429.940000,429.35000,429.905000,346721,MSFT
6,2026-01-30 15:57:00,429.670010,429.980010,429.53000,429.690000,293137,MSFT
7,2026-01-30 15:56:00,429.340000,429.706390,429.23999,429.679990,283368,MSFT
8,2026-01-30 15:59:00,338.100010,338.450010,337.92001,338.230010,601720,GOOGL
9,2026-01-30 15:58:00,338.019989,338.130000,337.94501,338.100010,215059,GOOGL


### Loading

In [17]:
# database credentials
load_dotenv()
DB_USER = os.getenv('DB_USER')
DB_PASSWORD = os.getenv('DB_PASSWORD')
DB_HOST = os.getenv('DB_HOST')
DB_NAME = os.getenv('DB_NAME')
DB_PORT = os.getenv('DB_PORT')

In [18]:
DB_NAME

'apex_db'

In [19]:
#create connection string with url
db_url = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine = create_engine(db_url)

# load data to data base 
all_df.to_sql('stockprices_data',engine,if_exists='append',index=False)

print('data loaded successfully')

data loaded successfully


In [20]:
# save to csv
all_df.to_csv('stockprice_data.csv',index=False)